In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import kagglehub
import datasethelpers as dshlep
import torch
import torchvision

In [4]:
path = kagglehub.dataset_download("orvile/cbis-ddsm-dataset", output_dir="./data-cleaned")

100%|██████████| 2.85G/2.85G [03:03<00:00, 16.6MB/s]

Extracting files...


In [9]:
# download the CBIS-DDSM dataset from kaggle

output_dir = "./data"
path = kagglehub.dataset_download("awsaf49/cbis-ddsm-breast-cancer-image-dataset", output_dir=output_dir)
print(path)

100%|██████████| 4.95G/4.95G [06:07<00:00, 14.5MB/s]

Extracting files...


./data


In [40]:
path = "./data"
mass_ds_train = pd.read_csv(os.path.join(path, "csv/mass_case_description_train_set.csv"))
calc_ds_train = pd.read_csv(os.path.join(path, "csv/calc_case_description_train_set.csv"))
dicom_df = pd.read_csv(os.path.join(path, "csv/dicom_info.csv"))
# count the number of benign and malignant cases
print(mass_ds_train["pathology"].value_counts())
print("Total mass cases:", len(mass_ds_train))
print("Total calc cases:", len(calc_ds_train))
print("Total dicom cases:", len(dicom_df))

valid_descriptions = {"cropped images", "ROI mask images", "full mammogram images"}

filtered_df = dicom_df[dicom_df['SeriesDescription'].isin(valid_descriptions)]
print(filtered_df['SeriesDescription'].value_counts())

pathology
MALIGNANT                  637
BENIGN                     577
BENIGN_WITHOUT_CALLBACK    104
Name: count, dtype: int64
Total mass cases: 1318
Total calc cases: 1546
Total dicom cases: 10237
SeriesDescription
cropped images           3567
ROI mask images          3247
full mammogram images    2857
Name: count, dtype: int64


In [47]:

row = filtered_df.iloc[1]
series_desc = row['SeriesDescription'] 
patient_id_composite = row['PatientID'] 
print(row)

BASE_INPUT_PATH = "./data/"
CSV_FOLDER_PATH = os.path.join(BASE_INPUT_PATH, "csv")
IMAGE_FOLDER_PATH = os.path.join(BASE_INPUT_PATH, "jpeg")

full_path = None
if 'image_path' in row and pd.notna(row['image_path']):
    rel_path = row['image_path']

    # if series_desc is "full mammogram images":

    if 'jpeg' in rel_path:
        clean_rel_path = rel_path.split('jpeg')[-1].strip("/\\")
        full_path = os.path.join(IMAGE_FOLDER_PATH, clean_rel_path)
    else:
        full_path = os.path.join(IMAGE_FOLDER_PATH, rel_path)

print(full_path)

file_path                                      CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.24838...
image_path                                     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.248386...
AccessionNumber                                                                              NaN
BitsAllocated                                                                                 16
BitsStored                                                                                    16
BodyPartExamined                                                                          BREAST
Columns                                                                                     3526
ContentDate                                                                             20160426
ContentTime                                                                           143829.101
ConversionType                                                                               WSD
HighBit                       

In [46]:
img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
print(img.shape)

(6256, 3526)


In [ ]:
print(mass_ds_train["assessment"].value_counts())

assessment
4    533
5    299
3    279
0    129
2     77
1      1
Name: count, dtype: int64


In [6]:
# combine benign and benign without callback into a single "benign" category
mass_ds_train["pathology"] = mass_ds_train["pathology"].replace({"BENIGN_WITH_CALLBACK": "BENIGN"})

mass_imgs = dshlep.getJPGTensorsFromDS(mass_ds_train, path, "mass_images")
calc_imgs = dshlep.getJPGTensorsFromDS(calc_ds_train, path, "calc_images")

  0%|          | 0/1318 [00:00<?, ?it/s]

TypeError: zeros(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got str"

In [ ]:
print(torchvision.__version__)
print(torch.cuda.is_available())


0.21.0+cu124


In [5]:
BASE_CLEAN_PATH = "./data-cleaned/"

# numfiles in the base clean path
num_files = len(os.listdir(os.path.join(BASE_CLEAN_PATH,'train','ann')))
print(f"Num annotation files: {num_files}")
num_files = len(os.listdir(os.path.join(BASE_CLEAN_PATH,'train','img')))
print(f"Num image files: {num_files}")

Num annotation files: 2458
Num image files: 237


In [6]:
a = "Mass"
print('MASS' in str(a).upper())


True
